# 2.3 GPU workloads as agent tools

Follows [03-gpu-tools.md](../03-gpu-tools.md). The agent runs here, on your laptop's CPU. The
image tool calls the Sana Endpoint on a GPU. Same tool signature as 2.2, real Endpoint behind it.

Before you run: start the Sana Endpoint from its
[template](https://console.nebius.com/serverless/endpoint/create?template=sana). It bills while
`RUNNING`: the last cell is the teardown.

`MODE = "live"`: Token Factory LLM and the Sana Endpoint. `"fake"`: scripted model, placeholder
images, no network.

In [ ]:
MODE = "live"  # "live" or "fake"

In [ ]:
import sys
from pathlib import Path

MODULE = "02-multi-agent-media-workflows"
LESSON = "03-gpu-tools"  # runs go to code/runs/03-gpu-tools/<run_id>/
candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd() / MODULE / "code"]
CODE = next(p for p in candidates if (p / "tools.py").exists())
sys.path.insert(0, str(CODE))
ASSETS = CODE.parent / "assets"

In [ ]:
import os

from pydantic_ai import Agent

from fake import PlaceholderImages, block_real_models, call, scripted_model
from models import Brief
from tools import (
    SanaEndpoint,
    Store,
    ToolDeps,
    generate_image,
    llm_model,
    optional_setting,
    settings,
)
from tracing import Tracer, new_run_id, run_traced

brief = Brief.from_yaml(ASSETS / "brief.yaml")
GPU_PRICE_PER_HOUR = None  # USD/h of the Endpoint's GPU and preset, from the pricing page

## Credentials stay on the tool side

The Endpoint URL, token and ID go to `code/.env` (mode 600). The tool reads them; the model,
the prompt and the trace never contain them.

In [ ]:
if MODE == "live":
    env = settings("SANA_BASE_URL", "SANA_TOKEN", "SANA_ENDPOINT_ID", secret=("SANA_TOKEN",))
    images = SanaEndpoint(env["SANA_BASE_URL"], env["SANA_TOKEN"], env["SANA_ENDPOINT_ID"])
    print(f"Endpoint ready after {images.wait_ready():.0f} s of waiting")
    model = llm_model()
else:
    block_real_models()
    images = PlaceholderImages()
    model = scripted_model(call("generate_image", prompt=brief.shot), "Keyframe ready.")

tracer = Tracer(new_run_id(), LESSON)
deps = ToolDeps(
    tracer, images, Store(tracer.folder, bucket=optional_setting("BUCKET")), GPU_PRICE_PER_HOUR
)

## Default path: Sana as generate_image

In [ ]:
agent = Agent(
    model,
    deps_type=ToolDeps,
    tools=[generate_image],
    instructions="You make one keyframe image for a video ad brief. Call generate_image once.",
)
result = await run_traced(agent, f"Brief: {brief.model_dump_json()}", tracer=tracer, deps=deps)
print(result.output)
for e in tracer.events():
    print(
        f"{e.kind:14} {e.workload:9} {e.workload_id or '':32} t_gen_s={e.t_gen_s} cost={e.cost_usd:.5f}"
    )

In [ ]:
from IPython.display import Image

keyframe = next(tracer.folder.glob("keyframe-*.png"))
if MODE == "live":
    assert env["SANA_TOKEN"] not in tracer.path.read_text(), "token leaked into the trace"
Image(filename=keyframe, width=416)

Optional, for the timeout in 2.2 and the budget estimate in 2.5: generation time over 10 seeds.

In [ ]:
import statistics

from tools import image_asset

MEASURE = False  # True: 10 more generations on the Endpoint
if MODE == "live" and MEASURE:
    times = []
    for seed in range(100, 110):
        times.append((await image_asset(images, deps.store, brief.shot, seed, "832x480")).t_gen_s)
    print(
        f"median {statistics.median(times):.2f} s, p95 {statistics.quantiles(times, n=20)[-1]:.2f} s"
    )

## Also try: the same tool over MCP

`mcp_server.py` serves `generate_image` over stdio. The agent connects to it as a toolset.
The Endpoint token lives in the server process. Our hook writes the trace event on the client
side, where the tool call ID is known.

In [ ]:
import json

from pydantic_ai.mcp import MCPServerStdio

from tools import AssetRef, trace_asset


async def trace_mcp_call(ctx, call_tool, name, tool_args):
    """Call the MCP tool, then write the tool_call event with the framework's call ID."""
    result = await call_tool(name, tool_args, None)
    asset = AssetRef.model_validate(result if isinstance(result, dict) else json.loads(result))
    trace_asset(deps, ctx.tool_call_id, name, asset)
    return result


server_env = {
    **os.environ,
    "RUN_ID": tracer.run_id,
    **({} if MODE == "live" else {"MCP_FAKE": "1"}),
}
server = MCPServerStdio(
    "uv",
    args=["run", "python", "mcp_server.py"],
    cwd=CODE,
    env=server_env,
    process_tool_call=trace_mcp_call,
)
model = (
    llm_model()
    if MODE == "live"
    else scripted_model(call("generate_image", prompt=brief.shot, seed=8), "Done.")
)
mcp_agent = Agent(model, deps_type=ToolDeps, toolsets=[server])
async with mcp_agent:
    result = await run_traced(
        mcp_agent, f"Brief: {brief.model_dump_json()}. Use seed 8.", tracer=tracer, deps=deps
    )
print(result.output)

## Teardown

Stop the Endpoint now: in the console, open it and click **Stop**. Then check that nothing is
still `RUNNING`:

```bash
nebius ai endpoint list
```